# Autoship Nudge Promo Incentive — Data Analysis (Subsequent Fix Order Rate)

This notebook builds up, CTE by CTE, the warehouse query used to establish the Subsequent Fix Order Rate baseline in `power_analysis_sfo_v2.ipynb`. Each step below adds one CTE and re-runs, so the final step reproduces the exact query and numbers used for sizing.

**Population:** Manual clients (i.e., not already enrolled in Autoship) who completed First Fix checkout with a Buy 1+ keep rate — the eligible population for the Autoship Nudge Promo Incentive Test, per the experiment's PRD.

**Source table:** `curated.merch_sales_and_feedback`, an item-level Fix/direct-buy fact table. `autoship_or_manual` and `fix_number` identify each client's First Fix and keep rate; `fix_number >= 2` identifies whether and when a client placed a subsequent order, regardless of how it was fulfilled.

In [1]:
import numpy as np
import pandas as pd
from amphibian import get_data_accessor

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

def query(sql):
    return get_data_accessor(engines=['presto']).fetch_sql(sql=sql)

## Part A — Identifying each client's First Fix

`curated.merch_sales_and_feedback` is at the item grain (one row per item shipped), so a client's First Fix (`fix_number = 1`) spans multiple rows. This part collapses those rows to one row per client's first-Fix shipment, carrying forward the shipment's `autoship_or_manual` value and the count of items kept.

### A1 — raw item-level rows, `fix_number = 1`

In [2]:
query("""--sql
SELECT client_id, shipment_id, item_id, checkout_date, autoship_or_manual, sold_paid_fix_flag, business_line
FROM curated.merch_sales_and_feedback
WHERE fix_number = 1
  AND created_date >= DATE '2026-06-01'
ORDER BY client_id, shipment_id
LIMIT 8
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,client_id,shipment_id,item_id,checkout_date,autoship_or_manual,sold_paid_fix_flag,business_line
0,3008420,134765685,379180269,2026-07-05,autoship,0,Womens
1,3008420,134765685,387578192,2026-07-05,autoship,0,Womens
2,3008420,134765685,372520741,2026-07-05,autoship,0,Womens
3,3008420,134765685,388218846,2026-07-05,autoship,0,Womens
4,3008420,134765685,388424823,2026-07-05,autoship,0,Womens
5,3010612,134862228,385837912,2026-07-11,autoship,0,Womens
6,3010612,134862228,381294500,2026-07-11,autoship,0,Womens
7,3010612,134862228,375364919,2026-07-11,autoship,0,Womens


Each client's first-Fix shipment shows up as several rows (one per item). `autoship_or_manual` is constant within a shipment (it describes how the whole Fix was scheduled, not the item), so it collapses cleanly with `ARBITRARY()`; `sold_paid_fix_flag` is summed per shipment to get the number of items kept.

### A2 — collapse to one row per (client, shipment), and check cardinality

In [3]:
query("""--sql
WITH first_fix AS (
    SELECT
        client_id,
        shipment_id,
        MIN(checkout_date) AS checkout_date,
        ARBITRARY(autoship_or_manual) AS autoship_or_manual,
        SUM(sold_paid_fix_flag) AS n_items_kept
    FROM curated.merch_sales_and_feedback
    WHERE fix_number = 1
      AND created_date >= DATE '2026-01-01'
    GROUP BY client_id, shipment_id
)
SELECT n_shipments, COUNT(*) AS n_clients
FROM (
    SELECT client_id, COUNT(DISTINCT shipment_id) AS n_shipments
    FROM first_fix
    GROUP BY client_id
)
GROUP BY n_shipments
ORDER BY n_shipments
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,n_shipments,n_clients
0,1,391727
1,2,542
2,3,17
3,4,7
4,6,1


The overwhelming majority of clients have exactly one shipment tagged `fix_number = 1`; a small minority (data-entry edge cases, likely household or correction records) have 2+. The final query below keeps only the chronologically earliest such shipment per client via `ROW_NUMBER() OVER (PARTITION BY client_id ORDER BY checkout_date)`, so every eligible client contributes exactly one row.

## Part B — Manual vs. Autoship split, and why a recent cohort is used

`autoship_or_manual` marks whether a Fix was fulfilled under an active Autoship subscription at the time. Historically, clients could enroll in Autoship at signup, before ever receiving a Fix — so a meaningful share of *First* Fixes were historically already `'autoship'`. A more recent rollout moved Autoship enrollment to **after** First Fix checkout, once keep rate is known, via the same post-checkout nudge moment this experiment is testing. That shifts the manual share of First Fixes upward over time, which is why the baseline later in this notebook uses a recent reference month rather than a long historical average.

In [4]:
query("""--sql
SELECT
    DATE_TRUNC('month', created_date) AS month,
    autoship_or_manual,
    COUNT(DISTINCT client_id) AS n_clients
FROM curated.merch_sales_and_feedback
WHERE fix_number = 1
  AND created_date >= DATE '2025-08-01'
GROUP BY 1, 2
ORDER BY 1 DESC, 2
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,month,autoship_or_manual,n_clients
0,2026-08-01,autoship,5697
1,2026-08-01,manual,6541
2,2026-07-01,autoship,24579
3,2026-07-01,manual,28473
4,2026-06-01,autoship,20647
5,2026-06-01,manual,19868
6,2026-05-01,autoship,36903
7,2026-05-01,manual,12892
8,2026-04-01,autoship,42355
9,2026-04-01,manual,14193


The manual share jumps sharply from ~23-26% (Jan-May 2026) to ~49-54% (Jun-Aug 2026), the rollout visibly maturing partway through this window. That has a direct consequence for sizing: the eligible **daily volume** (how many Manual, Buy 1+ First-Fix clients show up per day) roughly doubles once the rollout is fully in effect, so a volume figure measured from *before* that jump would understate the true run rate this experiment will actually see. Unlike the order-rate read (which needs a 90-day maturation wait for the 2nd Fix to resolve), the eligible population itself is known immediately at First Fix checkout — so daily volume can be measured off the most recent complete month, decoupled from the maturation gate the order rate needs. See the dedicated volume read later in this notebook.

## Part C — Keep rate: Buy 0 vs. Buy 1+

The PRD scopes this test to clients with a **Buy 1+** keep rate on their First Fix (kept at least one item) — Buy 0 clients always get the BAU Quick Fix experience with no Autoship nudge at all, and are out of scope for this promo/non-promo comparison. `sold_paid_fix_flag`, summed per shipment, gives the number of items kept.

In [5]:
query("""--sql
WITH first_fix AS (
    SELECT client_id, shipment_id, SUM(sold_paid_fix_flag) AS n_items_kept
    FROM curated.merch_sales_and_feedback
    WHERE fix_number = 1
      AND created_date >= DATE '2026-06-01'
    GROUP BY client_id, shipment_id
)
SELECT n_items_kept, COUNT(*) AS n_shipments
FROM first_fix
GROUP BY n_items_kept
ORDER BY n_items_kept
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,n_items_kept,n_shipments
0,0,43433
1,1,12266
2,2,12155
3,3,9740
4,4,4848
5,5,14443
6,6,1574
7,7,997
8,8,1967
9,9,245


Roughly half to three-fifths of recent First Fixes keep at least one item — the Buy 1+ gate this test's population applies. This fix-level "kept at least one item" share is naturally higher than the commonly cited *item-level* keep rate, since Buy 1+ only requires one kept item out of a typical 5-item Fix.

## Part D — How long to wait for a "did they order a subsequent Fix" read

Whether a client ordered a subsequent Fix is read off whether they have any `fix_number >= 2` shipment: if that Fix checks out, the client ordered. Reading that signal requires waiting for the next Fix to actually happen. This step checks the typical gap between a client's first and second Fix checkout, to choose a maturation window.

In [6]:
query("""--sql
WITH ff1 AS (
    SELECT client_id, MIN(checkout_date) AS checkout_date_1
    FROM curated.merch_sales_and_feedback
    WHERE fix_number = 1 AND created_date >= DATE '2025-10-01' AND created_date < DATE '2026-03-01'
    GROUP BY client_id
),
ff2 AS (
    SELECT client_id, MIN(checkout_date) AS checkout_date_2
    FROM curated.merch_sales_and_feedback
    WHERE fix_number = 2 AND created_date >= DATE '2025-10-01'
    GROUP BY client_id
)
SELECT
    APPROX_PERCENTILE(DATE_DIFF('day', a.checkout_date_1, b.checkout_date_2), 0.5) AS median_gap_days,
    APPROX_PERCENTILE(DATE_DIFF('day', a.checkout_date_1, b.checkout_date_2), 0.75) AS p75_gap_days,
    APPROX_PERCENTILE(DATE_DIFF('day', a.checkout_date_1, b.checkout_date_2), 0.9) AS p90_gap_days,
    COUNT(*) AS n_with_2nd_fix,
    (SELECT COUNT(*) FROM ff1) AS n_with_1st_fix
FROM ff1 a
JOIN ff2 b ON a.client_id = b.client_id
WHERE b.checkout_date_2 > a.checkout_date_1
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,median_gap_days,p75_gap_days,p90_gap_days,n_with_2nd_fix,n_with_1st_fix
0,28,54,91,165811,259246


Median time to a second Fix is close to a typical 4-week cadence, with a long tail out past 90 days for slower re-orderers. A **90-day maturation window** is used throughout — it comfortably covers the bulk of clients who will get a second Fix reasonably promptly, and it conveniently mirrors the promo's own 90-day validity window from the PRD (Sec. 9), so a client's order-rate read and their promo's expiration are checked over the same horizon.

## Part E — The full baseline query

Putting it together: `eligible` applies the Manual + Buy 1+ gate (deduped to each client's earliest `fix_number = 1` shipment) and the maturation cutoff; `next_fix` finds each eligible client's earliest `fix_number >= 2` checkout, if any; the final `SELECT` flags `ordered_subsequent_fix` and aggregates to a monthly Subsequent Fix Order Rate and daily eligible volume. This is the exact query used in `power_analysis_sfo_v2.ipynb`.

In [7]:
MATURATION_DAYS = 90
COHORT_START = '2025-08-01'

baseline_query = f"""--sql
WITH first_fix AS (
    SELECT
        client_id,
        shipment_id,
        MIN(checkout_date) AS checkout_date,
        ARBITRARY(autoship_or_manual) AS autoship_or_manual,
        SUM(sold_paid_fix_flag) AS n_items_kept
    FROM curated.merch_sales_and_feedback
    WHERE fix_number = 1
      AND created_date >= DATE '{COHORT_START}'
    GROUP BY client_id, shipment_id
),
eligible AS (
    SELECT client_id, checkout_date
    FROM (
        SELECT client_id, checkout_date, autoship_or_manual, n_items_kept,
               ROW_NUMBER() OVER (PARTITION BY client_id ORDER BY checkout_date) AS rn
        FROM first_fix
    )
    WHERE rn = 1
      AND autoship_or_manual = 'manual'
      AND n_items_kept >= 1
      AND checkout_date <= CURRENT_DATE - INTERVAL '{MATURATION_DAYS}' DAY
),
next_fix AS (
    SELECT client_id, MIN(checkout_date) AS next_fix_checkout_date
    FROM curated.merch_sales_and_feedback
    WHERE fix_number >= 2
    GROUP BY client_id
),
joined AS (
    SELECT
        e.client_id,
        DATE_TRUNC('month', e.checkout_date) AS month,
        e.checkout_date,
        CASE WHEN n.next_fix_checkout_date IS NOT NULL
              AND n.next_fix_checkout_date <= e.checkout_date + INTERVAL '{MATURATION_DAYS}' DAY
             THEN 1 ELSE 0 END AS ordered_subsequent_fix
    FROM eligible e
    LEFT JOIN next_fix n ON n.client_id = e.client_id
),
month_days AS (
    SELECT month, COUNT(DISTINCT checkout_date) AS days_observed
    FROM joined
    GROUP BY month
)
SELECT
    j.month,
    md.days_observed,
    COUNT(*) AS n_eligible,
    SUM(ordered_subsequent_fix) AS n_ordered,
    CAST(SUM(ordered_subsequent_fix) AS DOUBLE) / COUNT(*) AS subsequent_fix_order_rate,
    ROUND(COUNT(*) / CAST(md.days_observed AS DOUBLE), 1) AS eligible_per_day
FROM joined j
JOIN month_days md ON j.month = md.month
GROUP BY j.month, md.days_observed
ORDER BY j.month DESC
"""

query(baseline_query)

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,month,days_observed,n_eligible,n_ordered,subsequent_fix_order_rate,eligible_per_day
0,2026-05-01,8,2678,1273,0.475355,334.8
1,2026-04-01,30,10261,4775,0.465354,342.0
2,2026-03-01,31,10857,4880,0.449480,350.2
3,2026-02-01,28,8800,4182,0.475227,314.3
4,2026-01-01,31,10366,4753,0.458518,334.4
5,2025-12-01,31,8915,4294,0.481660,287.6
6,2025-11-01,30,7248,3156,0.435430,241.6
7,2025-10-01,31,9201,3867,0.420280,296.8
8,2025-09-01,30,9272,3838,0.413934,309.1
9,2025-08-01,31,7167,3005,0.419283,231.2


**Reading this:** the most recent fully-mature calendar month (all its days past the 90-day maturation cutoff) is the reference month `power_analysis_sfo_v2.ipynb` uses for its **order-rate** baseline. Earlier months in this table are shown for trend context — note the manual share (and therefore this population's size) drifting as the rollout matures, per Part B. That drift means this same reference month is *not* a reliable read of current eligible **volume** — Part F below measures volume separately, from a more recent month.

## Part F — A fresher, unmatured daily-volume read

Daily eligible volume doesn't need the 90-day maturation wait the order-rate read needs — whether a client is Manual + Buy 1+ is known immediately at First Fix checkout, with no need to wait and see what their second Fix looks like. That means volume can be measured off the **most recent complete month**, even though that same month is too recent to supply a matured order-rate read. This query is the same `eligible` CTE as Part E, minus the maturation cutoff.

In [8]:
query("""--sql
WITH first_fix AS (
    SELECT
        client_id,
        shipment_id,
        MIN(checkout_date) AS checkout_date,
        ARBITRARY(autoship_or_manual) AS autoship_or_manual,
        SUM(sold_paid_fix_flag) AS n_items_kept
    FROM curated.merch_sales_and_feedback
    WHERE fix_number = 1
      AND created_date >= DATE '2026-05-01'
    GROUP BY client_id, shipment_id
),
eligible AS (
    SELECT client_id, checkout_date
    FROM (
        SELECT client_id, checkout_date, autoship_or_manual, n_items_kept,
               ROW_NUMBER() OVER (PARTITION BY client_id ORDER BY checkout_date) AS rn
        FROM first_fix
    )
    WHERE rn = 1
      AND autoship_or_manual = 'manual'
      AND n_items_kept >= 1
),
month_days AS (
    SELECT DATE_TRUNC('month', checkout_date) AS month, COUNT(DISTINCT checkout_date) AS days_observed
    FROM eligible
    GROUP BY 1
)
SELECT
    DATE_TRUNC('month', e.checkout_date) AS month,
    md.days_observed,
    COUNT(*) AS n_eligible,
    ROUND(COUNT(*) / CAST(md.days_observed AS DOUBLE), 1) AS eligible_per_day
FROM eligible e
JOIN month_days md ON DATE_TRUNC('month', e.checkout_date) = md.month
GROUP BY DATE_TRUNC('month', e.checkout_date), md.days_observed
ORDER BY 1 DESC
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,month,days_observed,n_eligible,eligible_per_day
0,2026-08-01,6,3318,553.0
1,2026-07-01,31,17748,572.5
2,2026-06-01,30,12150,405.0
3,2026-05-01,31,6820,220.0
4,2026-04-01,1,10,10.0


**Reading this:** July 2026 (the most recent complete calendar month) runs meaningfully higher than April's matured-cohort volume — consistent with the manual-share jump in Part B. `power_analysis_sfo_v2.ipynb` uses July's volume for duration calculations, decoupled from April's rate, rather than understating the run rate with a stale volume figure.